# Hermes + 9Router di Kaggle

Kaggle dipakai sebagai runtime notebook sementara, bukan host 24/7. Hermes berjalan di sesi Kaggle dan mengirim permintaan ke 9Router yang sudah dideploy terpisah dengan HTTPS. Sesi Kaggle dapat berhenti dan filesystem-nya dapat direset.

Di Kaggle, aktifkan **Internet** dan tambahkan Secrets berikut: `NINE_ROUTER_BASE_URL` (origin HTTPS tanpa `/v1`), `NINE_ROUTER_API_KEY`, dan opsional `HERMES_MODEL`. Jangan menaruh key di output/cell notebook.

Unggah `kaggle_hermes_9router.py` sebagai Kaggle Dataset, lalu attach dataset itu ke notebook. Pastikan 9Router memakai `REQUIRE_API_KEY=true` dan endpoint publik HTTPS. Notebook hanya membuka Hermes API di `127.0.0.1`, bukan ke internet.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, tempfile
import requests

if shutil.which('hermes') is None:
    installer_response = requests.get('https://hermes-agent.nousresearch.com/install.sh', timeout=60)
    installer_response.raise_for_status()
    installer_path = Path(tempfile.gettempdir()) / 'hermes-install.sh'
    installer_path.write_bytes(installer_response.content)
    subprocess.run(['bash', str(installer_path)], check=True)

for candidate in (Path.home() / '.local/bin', Path.home() / '.hermes/bin'):
    if candidate.is_dir():
        os.environ['PATH'] = f"{candidate}:{os.environ['PATH']}"
if shutil.which('hermes') is None:
    raise RuntimeError('Hermes belum ditemukan pada PATH. Periksa output installer dan dokumentasi instalasi Hermes.')
print(subprocess.run(['hermes', '--version'], check=True, text=True, capture_output=True).stdout.strip())

In [ ]:
import runpy

entry_candidates = list(Path('/kaggle/input').glob('**/kaggle_hermes_9router.py'))
local_entry = Path('kaggle_hermes_9router.py')
if local_entry.is_file():
    entry_candidates.insert(0, local_entry)
if not entry_candidates:
    raise FileNotFoundError('Attach Kaggle Dataset berisi kaggle_hermes_9router.py ke notebook ini.')
runpy.run_path(str(entry_candidates[0]))

## Catatan sesi

Launcher mengakhiri proses Hermes setelah satu prompt uji. Jalankan ulang cell untuk sesi baru. Jangan mengekspos port `8642`; bila sesi Kaggle berakhir, Hermes ikut berhenti. Untuk bot atau gateway 24/7, jalankan Hermes pada host container persisten dan biarkan Kaggle hanya sebagai lingkungan uji.